# Notebook 07: Verify & Audit Dataset V2 & Huấn Luyện Thực Nghiệm Trên Google Colab

> **Mục tiêu của Notebook:**
> 1. **Giai đoạn 1 (Audit & Verification):** Kết nối Drive, kiểm tra checksum SHA256 cho cả 3 bộ split (80/10/10, 70/15/15, 60/20/20), khảo sát cấu trúc thư mục bên trong file ZIP, kiểm tra tính toàn vẹn của cặp ảnh/nhãn và class IDs (0..47), tự động chuẩn hóa file `data.yaml` tương thích với YOLO11.
> 2. **Giai đoạn 2 (Training & Evaluation):** Cho phép chọn nhanh tỉ lệ split (ví dụ: `70_15_15`), giải nén siêu tốc lên SSD Colab và chạy thực nghiệm Baseline + CBAM (hỗ trợ lưu checkpoint lên Drive và tự khôi phục khi bị ngắt kết nối/crash).

---

## 📌 Bước 1️⃣: Kết Nối Google Drive & Chuẩn Bị Môi Trường
> **Yêu cầu trước khi chạy:**
> - Tạo lối tắt (Shortcut) của thư mục chứa dataset hoặc thư mục chung `KLTN_38` / `KLTN-2026-testingNtraining` vào **My Drive** của bạn.

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Các khả năng đường dẫn thư mục dataset-v2 trên Drive (tự động nhận diện)
POSSIBLE_DATASET_PATHS = [
    "/content/drive/MyDrive/KLTN_38/dataset/dataset-v2",
    "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/dataset-v2",
    "/content/drive/MyDrive/dataset/dataset-v2",
    "/content/drive/MyDrive/dataset-v2"
]

DATASET_V2_DIR = None
for p in POSSIBLE_DATASET_PATHS:
    if os.path.exists(p):
        DATASET_V2_DIR = p
        break

if DATASET_V2_DIR is None:
    print("⚠️ Chưa tự động tìm thấy thư mục dataset-v2 theo đường dẫn mặc định.")
    print("👉 Hãy kiểm tra lại tên lối tắt trên Drive và gán thủ công DATASET_V2_DIR bên dưới.")
    # Bạn có thể gán thủ công nếu đường dẫn khác:
    DATASET_V2_DIR = "/content/drive/MyDrive/KLTN_38/dataset/dataset-v2"
else:
    print(f"✅ Đã tìm thấy thư mục Dataset V2 tại: {DATASET_V2_DIR}")

# Clone repo mã nguồn
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git /content/klcn2026
%cd /content/klcn2026
!git checkout main
!git pull origin main

# Cài đặt thư viện
!pip install -q ultralytics albumentations tabulate pyyaml
print("\n🚀 Môi trường đã sẵn sàng!")

## 🔍 Bước 2️⃣: GIAI ĐOẠN 1 — Kiểm Tra Toàn Vẹn SHA256 & Khảo Sát Tệp ZIP
> Cell này sẽ quét cả 3 thư mục `dataset-v2-80/10/10`, `dataset-v2-70/15/15`, `dataset-v2-60/20/20` để:
> 1. Tính toán mã băm SHA256 thực tế của file `.zip` và đối chiếu với file `.sha256` đi kèm.
> 2. Đọc lướt danh sách tệp bên trong ZIP (không tốn dung lượng SSD) để xác định cấu trúc: dạng `images/train` hay dạng `.txt`.

In [ ]:
import hashlib
import zipfile
from tabulate import tabulate

SPLIT_CONFIGS = [
    {
        "name": "80/10/10",
        "folder": "dataset-v2-80/10/10",
        "zip_name": "KLTN_prep_v2_80_10_10.zip",
        "sha_name": "KLTN_prep_v2_80_10_10.zip.sha256"
    },
    {
        "name": "70/15/15",
        "folder": "dataset-v2-70/15/15",
        "zip_name": "KLTN_prep_v2_70_15_15.zip",
        "sha_name": "KLTN_prep_v2_70_15_15.zip.sha256"
    },
    {
        "name": "60/20/20",
        "folder": "dataset-v2-60/20/20",
        "zip_name": "KLTN_prep_v2_60_20_20.zip",
        "sha_name": "KLTN_prep_v2_60_20_20.zip.sha256"
    }
]

def calculate_sha256(filepath: str) -> str:
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(65536 * 16), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

audit_summary = []
zip_structures = {}

print("=" * 80)
print("🔍 BẮT ĐẦU KIỂM TRA SHA256 VÀ KHẢO SÁT FILE ZIP TRÊN GOOGLE DRIVE")
print("=" * 80)

for cfg in SPLIT_CONFIGS:
    folder_path = os.path.join(DATASET_V2_DIR, cfg["folder"])
    zip_path = os.path.join(folder_path, cfg["zip_name"])
    sha_path = os.path.join(folder_path, cfg["sha_name"])

    if not os.path.exists(zip_path):
        print(f"❌ Không tìm thấy file zip: {zip_path}")
        audit_summary.append([cfg["name"], "MISSING", "N/A", "N/A", "❌ Không tìm thấy zip"])
        continue

    # Đọc expected sha256
    expected_sha = ""
    if os.path.exists(sha_path):
        with open(sha_path, "r", encoding="utf-8") as f:
            expected_sha = f.read().strip().split()[0].lower()

    print(f"\n⏳ Đang tính SHA256 cho {cfg['zip_name']} (~500MB)... Vui lòng đợi.")
    actual_sha = calculate_sha256(zip_path).lower()
    sha_status = "✅ KHỚP" if actual_sha == expected_sha else f"❌ LỆCH (Exp: {expected_sha[:8]}.. Act: {actual_sha[:8]}..)"

    # Khảo sát cấu trúc bên trong zip
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            namelist = z.namelist()
            total_files = len(namelist)
            has_images = any("images/" in n for n in namelist)
            has_labels = any("labels/" in n for n in namelist)
            has_yaml = any(n.endswith(".yaml") or n.endswith(".yml") for n in namelist)
            has_txt = any(n.endswith(".txt") and ("train" in n or "val" in n) for n in namelist)

            # Đếm ảnh theo split
            train_imgs = sum(1 for n in namelist if "images/" in n and "train" in n and not n.endswith('/'))
            val_imgs = sum(1 for n in namelist if "images/" in n and "val" in n and not n.endswith('/'))
            test_imgs = sum(1 for n in namelist if "images/" in n and "test" in n and not n.endswith('/'))

            zip_structures[cfg["name"]] = {
                "namelist_sample": namelist[:10],
                "train_imgs": train_imgs,
                "val_imgs": val_imgs,
                "test_imgs": test_imgs,
                "has_yaml": has_yaml,
                "has_txt": has_txt
            }

            counts_str = f"Train: {train_imgs} | Val: {val_imgs} | Test: {test_imgs}" if (train_imgs or val_imgs or test_imgs) else f"{total_files} files"
            audit_summary.append([cfg["name"], "OK", sha_status, counts_str, "yaml: " + str(has_yaml) + " | txt: " + str(has_txt)])
    except Exception as e:
        audit_summary.append([cfg["name"], "CORRUPT", actual_sha[:8], "Error", str(e)])

print("\n" + "=" * 80)
print("📊 BẢNG TỔNG HỢP KIỂM ĐỊNH TÍNH TOÀN VẸN DATASET V2")
print("=" * 80)
headers = ["Tỉ Lệ", "File Zip", "SHA256 Checksum", "Số Lượng Mẫu (Ảnh)", "Cấu Trúc Phát Hiện"]
print(tabulate(audit_summary, headers=headers, tablefmt="grid"))

print("\n📁 10 File mẫu đầu tiên trong mỗi zip:")
for k, v in zip_structures.items():
    print(f"\n-- [{k}] --")
    for sample in v["namelist_sample"]:
        print(f"   {sample}")

## 🧪 Bước 3️⃣: Giải Nén Thử Nghiệm & Audit Chi Tiết Nhãn
> Chọn 1 bộ dataset (mặc định: `70_15_15`) để giải nén lên ổ cứng SSD cục bộ của Colab (`/content/dataset_v2/`).
> Sau đó chạy script kiểm tra:
> - Mọi ảnh đều có file nhãn `.txt` tương ứng (trừ ảnh âm tính nếu có).
> - Class ID trong nhãn có hợp lệ (nằm trong `0..47`).
> - Tọa độ Bounding Box `[x_center, y_center, width, height]` có nằm chuẩn trong `[0, 1]`.

In [ ]:
import yaml
from glob import glob

# Chọn tỉ lệ muốn verify & dùng để huấn luyện:
SELECTED_RATIO = "70_15_15"  # Tùy chọn: '80_10_10', '70_15_15', '60_20_20'

RATIO_MAP = {
    "80_10_10": {"folder": "dataset-v2-80/10/10", "zip": "KLTN_prep_v2_80_10_10.zip"},
    "70_15_15": {"folder": "dataset-v2-70/15/15", "zip": "KLTN_prep_v2_70_15_15.zip"},
    "60_20_20": {"folder": "dataset-v2-60/20/20", "zip": "KLTN_prep_v2_60_20_20.zip"}
}

TARGET_ZIP = os.path.join(DATASET_V2_DIR, RATIO_MAP[SELECTED_RATIO]["folder"], RATIO_MAP[SELECTED_RATIO]["zip"])
EXTRACT_DIR = f"/content/dataset_v2_{SELECTED_RATIO}"

print(f"📦 Đang giải nén {TARGET_ZIP} vào {EXTRACT_DIR}...")
!mkdir -p "{EXTRACT_DIR}"
!unzip -q "{TARGET_ZIP}" -d "{EXTRACT_DIR}"
print(f"✅ Giải nén hoàn tất vào {EXTRACT_DIR}!")

# Xem cấu trúc các thư mục vừa giải nén
print("\n📂 Cấu trúc thư mục mức 1:")
!ls -la "{EXTRACT_DIR}"

# Chạy kiểm tra cặp ảnh / nhãn và Class ID
all_label_files = glob(f"{EXTRACT_DIR}/**/labels/**/*.txt", recursive=True)
if not all_label_files:
    all_label_files = glob(f"{EXTRACT_DIR}/**/*.txt", recursive=True)

print(f"\n🔍 Tìm thấy tổng cộng {len(all_label_files)} file nhãn txt.")

invalid_classes = set()
invalid_bboxes = 0
total_objects = 0

for lf in all_label_files:
    with open(lf, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                total_objects += 1
                cls_id = int(parts[0])
                if cls_id < 0 or cls_id >= 48:
                    invalid_classes.add(cls_id)
                coords = [float(x) for x in parts[1:5]]
                if any(c < 0.0 or c > 1.0 for c in coords):
                    invalid_bboxes += 1

print(f"📊 Tổng số bounding boxes: {total_objects}")
if invalid_classes:
    print(f"❌ CẢNH BÁO: Phát hiện Class ID nằm ngoài dải 0..47: {invalid_classes}")
else:
    print("✅ Toàn bộ Class ID đều hợp lệ trong khoảng 0..47!")

if invalid_bboxes > 0:
    print(f"⚠️ Có {invalid_bboxes} bbox có tọa độ vượt ngưỡng [0, 1].")
else:
    print("✅ Toàn bộ tọa độ Bounding Box đều chuẩn hóa đúng trong [0, 1]!")

## 🛠️ Bước 4️⃣: Tự Động Tạo / Chuẩn Hóa File `data.yaml` Tuyệt Đối
> Để YOLO11 không bị lỗi không tìm thấy ảnh hoặc sai lệch đường dẫn giữa Drive/Colab, Cell này sẽ:
> 1. Tự động tìm thư mục `images/train`, `images/val`, `images/test` hoặc danh sách `.txt` trong dữ liệu vừa giải nén.
> 2. Sinh ra file cấu hình `configs/data/exp_2/data_colab_active.yaml` với đường dẫn chuẩn xác nhất.

In [ ]:
import yaml
from pathlib import Path

# Đọc 48 class names chuẩn của đề tài
REF_YAML = "/content/klcn2026/configs/data/exp_2/data_70_15_15.yaml"
with open(REF_YAML, "r", encoding="utf-8") as f:
    ref_data = yaml.safe_load(f)
class_names = ref_data.get("names", [])

# Xác định vị trí train/val/test trong thư mục vừa giải nén
extract_p = Path(EXTRACT_DIR)

# Trường hợp 1: Có thư mục images/{train,val,test}
if (extract_p / "images" / "train").exists():
    train_path = str(extract_p / "images" / "train")
    val_path = str(extract_p / "images" / "val")
    test_path = str(extract_p / "images" / "test") if (extract_p / "images" / "test").exists() else val_path
# Trường hợp 2: Có các file train_*.txt hoặc train.txt
elif list(extract_p.glob("*train*.txt")):
    train_txt = list(extract_p.glob("*train*.txt"))[0]
    val_txt = list(extract_p.glob("*val*.txt"))[0]
    test_txt_list = list(extract_p.glob("*test*.txt"))
    test_txt = test_txt_list[0] if test_txt_list else val_txt
    train_path = str(train_txt)
    val_path = str(val_txt)
    test_path = str(test_txt)
# Trường hợp 3: Nằm trong 1 folder con cấp 1
elif any((p / "images" / "train").exists() for p in extract_p.iterdir() if p.is_dir()):
    sub_p = [p for p in extract_p.iterdir() if (p / "images" / "train").exists()][0]
    train_path = str(sub_p / "images" / "train")
    val_path = str(sub_p / "images" / "val")
    test_path = str(sub_p / "images" / "test") if (sub_p / "images" / "test").exists() else val_path
else:
    train_path = str(extract_p)
    val_path = str(extract_p)
    test_path = str(extract_p)

colab_yaml_content = {
    "path": EXTRACT_DIR,
    "train": train_path,
    "val": val_path,
    "test": test_path,
    "nc": len(class_names),
    "names": class_names
}

ACTIVE_YAML_PATH = "/content/klcn2026/configs/data/exp_2/data_colab_active.yaml"
with open(ACTIVE_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(colab_yaml_content, f, allow_unicode=True, sort_keys=False)

print(f"✅ Đã tạo file YAML cấu hình chuẩn tại: {ACTIVE_YAML_PATH}")
print("\n--- Nội dung YAML ---")
!cat {ACTIVE_YAML_PATH}

## 🚀 Bước 5️⃣: GIAI ĐOẠN 2 — Thực Nghiệm Huấn Luyện (Baseline & CBAM Variants)
> Cấu hình thư mục lưu kết quả trên Google Drive để không bị mất khi hết phiên Colab.

In [ ]:
# Tạo các folder lưu weights trên Google Drive
RUNS_DIR = f"/content/drive/MyDrive/KLTN-2026-testingNtraining/runs_exp2_{SELECTED_RATIO}"
os.makedirs(f"{RUNS_DIR}/baseline", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_backbone", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_neck", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_head", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_backbone_head", exist_ok=True)

print(f"✅ Thư mục lưu kết quả trên Drive: {RUNS_DIR}")

### Cell 5A: Huấn Luyện YOLO11n BASELINE (Exp 0)

In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/baseline" \
    --name train

### Cell 5B: Huấn Luyện CBAM BACKBONE (Exp 1)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_backbone" \
    --name train

### Cell 5C: Huấn Luyện CBAM NECK (Exp 2)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_neck.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_neck" \
    --name train

### Cell 5D: Huấn Luyện CBAM HEAD (Exp 4)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_head" \
    --name train

### Cell 5E: Huấn Luyện CBAM BACKBONE + HEAD (Biến Thể Phối Hợp)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_backbone_head" \
    --name train

### 🔄 Cell Tiếp Tục Khi Bị Crash (Resume)

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

# Điền đường dẫn checkpoint 'last.pt' cần tiếp tục huấn luyện:
RESUME_CHECKPOINT = f"{RUNS_DIR}/cbam_backbone/train/weights/last.pt"

if os.path.exists(RESUME_CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện từ: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    model.train(resume=True)
else:
    print(f"❌ Không tìm thấy checkpoint: {RESUME_CHECKPOINT}")